# 01_target_definition_v3.ipynb

## v2 대비 변경 사항
- 기존 v2 피처(16+6개) 전부 유지
- **마지막 N번 구매 간격을 개별 컬럼으로 펼친 피처 추가** (시퀀스 정보를 트리 모델이 읽을 수 있는 형태로 변환)
- 목적: LSTM/RNN 없이도 시퀀스 정보 자체가 성능 개선에 기여하는지 검증

## 목차
1~8. v2와 동일 (라벨 생성, 기존 22개 피처)
9. **[NEW] 마지막 N회 구매 간격 펼치기**
10. 데이터 결합 및 검증
11. 저장


## 1. 데이터 로드

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

trans     = pd.read_csv('../raw/transactions_train.csv', nrows=100000)
customers = pd.read_csv('../raw/customers.csv')
articles  = pd.read_csv('../raw/articles.csv')

print(f'Transactions : {trans.shape}')
print(f'Customers    : {customers.shape}')
print(f'Articles     : {articles.shape}')


Transactions : (100000, 5)
Customers    : (1371980, 7)
Articles     : (105542, 25)


## 2. 고객별 최대 구매 간격 계산

In [5]:
trans_all = pd.read_csv('../raw/transactions_train.csv', usecols=['t_dat', 'customer_id'])
trans_all['t_dat'] = pd.to_datetime(trans_all['t_dat'])

trans_sorted = trans_all.sort_values(['customer_id', 't_dat'])
trans_sorted['diff'] = trans_sorted.groupby('customer_id')['t_dat'].diff().dt.days
user_max_diff = trans_sorted.groupby('customer_id')['diff'].max().dropna()
print(f'구매 간격 계산 완료: {len(user_max_diff):,}명')


구매 간격 계산 완료: 1,230,767명


## 3. Cut-off 기준일 확정

In [6]:
q75_days_calc = user_max_diff.quantile(0.75)
q75_days      = int(q75_days_calc)
min_date      = trans_all['t_dat'].min()
max_date      = trans_all['t_dat'].max()
cut_off_date  = max_date - pd.Timedelta(days=q75_days)

print(f'cut_off_date : {cut_off_date.date()}')
print(f'관측 기간    : {q75_days}일')


cut_off_date : 2020-03-03
관측 기간    : 203일


## 4. 라벨(Churn) 생성

In [7]:
df_past_base   = trans_all[trans_all['t_dat'] <= cut_off_date]
df_future_base = trans_all[trans_all['t_dat'] >  cut_off_date]

user_stats = df_past_base.groupby('customer_id')['t_dat'].agg(['min','max']).reset_index()
user_stats.columns = ['customer_id','first_buy','last_buy_past']
user_stats['tenure'] = (cut_off_date - user_stats['first_buy']).dt.days

df_label = user_stats[user_stats['tenure'] >= 30].copy()

active_users_in_future = df_future_base['customer_id'].unique()
df_label['churn'] = df_label['customer_id'].isin(active_users_in_future).apply(lambda x: 0 if x else 1)
df_label = df_label[['customer_id','churn']]

print(f'분석 대상 고객 수: {len(df_label):,}명')
print(f'이탈률: {df_label["churn"].mean()*100:.2f}%')


분석 대상 고객 수: 1,143,652명
이탈률: 50.02%


## 5. 피처 생성 공통 로드

In [8]:
trans_feat = pd.read_csv('../raw/transactions_train.csv',
                         usecols=['t_dat','customer_id','article_id','price'])
trans_feat['t_dat'] = pd.to_datetime(trans_feat['t_dat'])
trans_feat = trans_feat[trans_feat['t_dat'] <= cut_off_date].copy()
print(f'피처 생성 대상 거래 수: {len(trans_feat):,}건')


피처 생성 대상 거래 수: 22,991,230건


### 5-1~5-9. 기존 피처 생성 (v2와 동일)

In [9]:
# RFM
user_rfm = trans_feat.groupby('customer_id').agg(
    frequency  = ('article_id','count'),
    price_mean = ('price','mean'),
    price_sum  = ('price','sum'),
    price_max  = ('price','max'),
).reset_index()

# Recency
user_last_buy = trans_feat.groupby('customer_id')['t_dat'].max().reset_index()
user_last_buy.columns = ['customer_id','last_buy_before_cutoff']
user_last_buy['recency'] = (cut_off_date - user_last_buy['last_buy_before_cutoff']).dt.days
user_last_buy = user_last_buy.drop(columns=['last_buy_before_cutoff'])

# 활동성
date_90d  = cut_off_date - pd.Timedelta(days=90)
date_180d = cut_off_date - pd.Timedelta(days=180)
freq_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['article_id'].count()
            .reset_index().rename(columns={'article_id':'frequency_last_90d'}))
freq_180d = (trans_feat[trans_feat['t_dat'] >= date_180d].groupby('customer_id')['article_id'].count()
             .reset_index().rename(columns={'article_id':'frequency_last_180d'}))

# 구매 간격
trans_sorted_feat = trans_feat.sort_values(['customer_id','t_dat'])
trans_sorted_feat['interval'] = trans_sorted_feat.groupby('customer_id')['t_dat'].diff().dt.days
user_interval = trans_sorted_feat.groupby('customer_id')['interval'].agg(
    avg_purchase_interval = 'mean', std_purchase_interval = 'std',
).reset_index()

# 다양성
trans_with_cat = pd.merge(trans_feat[['customer_id','article_id']],
                           articles[['article_id','product_group_name']],
                           on='article_id', how='left')
user_diversity = trans_with_cat.groupby('customer_id').agg(
    unique_articles       = ('article_id','nunique'),
    unique_product_groups = ('product_group_name','nunique'),
).reset_index()

# 취향
trans_style = pd.merge(trans_feat[['customer_id','article_id']],
                        articles[['article_id','index_group_name','product_group_name']],
                        on='article_id', how='left')
def get_most_frequent(x):
    if x.empty: return 'NONE'
    return x.value_counts().idxmax()
user_style = trans_style.groupby('customer_id').agg(
    index_group_name   = ('index_group_name',   get_most_frequent),
    product_group_name = ('product_group_name', get_most_frequent),
).reset_index()

print('기존 피처(RFM/Recency/활동성/간격/다양성/취향) 생성 완료')


기존 피처(RFM/Recency/활동성/간격/다양성/취향) 생성 완료


In [10]:
# [v2] 구매 트렌드 / 다양성 변화 / 구매 리듬
price_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['price'].mean()
             .reset_index().rename(columns={'price':'price_mean_last_90d'}))

user_trend = freq_90d.copy()
user_trend = pd.merge(user_trend, freq_180d, on='customer_id', how='outer')
user_trend = pd.merge(user_trend, price_90d, on='customer_id', how='left')
user_trend = pd.merge(user_trend, user_rfm[['customer_id','price_mean']], on='customer_id', how='left')
user_trend['frequency_last_90d']  = user_trend['frequency_last_90d'].fillna(0)
user_trend['frequency_last_180d'] = user_trend['frequency_last_180d'].fillna(0)
user_trend['freq_decay_rate'] = np.where(user_trend['frequency_last_180d'] > 0,
    user_trend['frequency_last_90d'] / user_trend['frequency_last_180d'], np.nan)
user_trend['recent_price_ratio'] = np.where(user_trend['price_mean'] > 0,
    user_trend['price_mean_last_90d'] / user_trend['price_mean'], np.nan)
user_trend = user_trend[['customer_id','freq_decay_rate','recent_price_ratio']]

diversity_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['article_id'].nunique()
                  .reset_index().rename(columns={'article_id':'unique_articles_last_90d'}))
user_diversity_change = pd.merge(diversity_90d, user_diversity[['customer_id','unique_articles']],
                                  on='customer_id', how='right')
user_diversity_change['unique_articles_last_90d'] = user_diversity_change['unique_articles_last_90d'].fillna(0)
user_diversity_change['diversity_decay_rate'] = np.where(user_diversity_change['unique_articles'] > 0,
    user_diversity_change['unique_articles_last_90d'] / user_diversity_change['unique_articles'], np.nan)
user_diversity_change = user_diversity_change[['customer_id','unique_articles_last_90d','diversity_decay_rate']]

last_interval_df = (trans_sorted_feat.groupby('customer_id')['interval'].last()
                     .reset_index().rename(columns={'interval':'last_interval'}))
user_rhythm = pd.merge(last_interval_df, user_interval[['customer_id','avg_purchase_interval']],
                        on='customer_id', how='left')
user_rhythm['interval_acceleration'] = np.where(user_rhythm['avg_purchase_interval'] > 0,
    user_rhythm['last_interval'] / user_rhythm['avg_purchase_interval'], np.nan)
user_rhythm = user_rhythm[['customer_id','last_interval','interval_acceleration']]

print('v2 신규 피처(트렌드/다양성변화/리듬) 생성 완료')


v2 신규 피처(트렌드/다양성변화/리듬) 생성 완료


## 9. [NEW] 마지막 N회 구매 간격 펼치기

기존 `interval_acceleration`은 '마지막 간격 / 평균 간격'이라는 **비율 하나**로 압축한 정보였습니다.
여기서는 압축하지 않고, **마지막 5번의 구매 간격을 개별 컬럼으로 그대로 펼쳐서** 모델에 전달합니다.

```
interval_t1  : 가장 최근 간격 (cut_off에 가장 가까운 간격)
interval_t2  : 그 이전 간격
interval_t3  : 그 이전
interval_t4  : 그 이전
interval_t5  : 가장 과거 간격
```

구매 횟수가 5회 미만인 고객은 해당 칸이 NaN으로 남습니다 (정보 없음 → 트리 모델 위임).
모두 cut_off 이전 데이터만 사용하므로 누수 없음.


In [11]:
N = 5  # 펼칠 구매 간격 개수

# 고객별로 시간순 정렬된 간격을 리스트로 모음
interval_lists = (
    trans_sorted_feat.dropna(subset=['interval'])
    .groupby('customer_id')['interval']
    .apply(list)
)

print(f'구매 간격 보유 고객 수: {len(interval_lists):,}명')
print(f'평균 보유 간격 개수: {interval_lists.apply(len).mean():.1f}개')


구매 간격 보유 고객 수: 1,050,786명
평균 보유 간격 개수: 20.8개


In [12]:
# 마지막 N개를 최신순(t1=가장 최근)으로 컬럼화
def get_last_n_intervals(interval_list, n=N):
    last_n = interval_list[-n:]      # 끝에서 n개 (시간순 정렬되어 있으므로 가장 최근 n개)
    last_n_reversed = last_n[::-1]   # 최신이 먼저 오도록 뒤집기 (t1=최신)
    padded = last_n_reversed + [np.nan] * (n - len(last_n_reversed))  # 부족하면 NaN 패딩
    return padded

interval_matrix = interval_lists.apply(get_last_n_intervals)
interval_df = pd.DataFrame(
    interval_matrix.tolist(),
    index=interval_matrix.index,
    columns=[f'interval_t{i+1}' for i in range(N)]
).reset_index()

print(f'시퀀스 피처 생성 완료: {interval_df.shape}')
interval_df.head()


시퀀스 피처 생성 완료: (1050786, 6)


,customer_id,interval_t1,interval_t2,interval_t3,interval_t4,interval_t5
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,6.0,55.0,0.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,36.0,0.0,0.0,0.0,0.0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,0.0,339.0,158.0,4.0
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.0,NaN,NaN,NaN,NaN
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,8.0,208.0,0.0,146.0,0.0


In [13]:
# 결측치 비율 확인 (구매 횟수가 적을수록 뒤 컬럼이 비게 됨 — 정상)
print('--- interval_t1~t5 결측치 비율(%) ---')
for col in [f'interval_t{i+1}' for i in range(N)]:
    null_pct = interval_df[col].isnull().mean() * 100
    print(f'{col} : {null_pct:.1f}%')


--- interval_t1~t5 결측치 비율(%) ---
interval_t1 : 0.0%
interval_t2 : 10.8%
interval_t3 : 18.9%
interval_t4 : 25.9%
interval_t5 : 31.4%


## 10. 데이터 결합 (Merge)

In [14]:
customers_features = customers[[
    'customer_id','FN','Active','club_member_status','fashion_news_frequency','age'
]]

df_analysis = df_label.copy()

for feat_df, name in [
    (user_rfm,              'RFM'),
    (user_last_buy,         'Recency'),
    (freq_90d,              '활동성 90일'),
    (freq_180d,             '활동성 180일'),
    (user_interval,         '구매 간격'),
    (user_diversity,        '다양성'),
    (user_style,            '취향'),
    (user_trend,            '[v2] 구매 트렌드'),
    (user_diversity_change, '[v2] 다양성 변화'),
    (user_rhythm,           '[v2] 구매 리듬'),
    (interval_df,           '[NEW] 시퀀스 간격 t1~t5'),
    (customers_features,    '고객 메타'),
]:
    df_analysis = pd.merge(df_analysis, feat_df, on='customer_id', how='left')
    print(f'{name} 병합 완료 → shape: {df_analysis.shape}')

df_analysis['frequency_last_90d']  = df_analysis['frequency_last_90d'].fillna(0).astype(int)
df_analysis['frequency_last_180d'] = df_analysis['frequency_last_180d'].fillna(0).astype(int)
df_analysis['unique_articles_last_90d'] = df_analysis['unique_articles_last_90d'].fillna(0).astype(int)

print(f'\n최종 데이터 크기: {df_analysis.shape}')


RFM 병합 완료 → shape: (1143652, 6)
Recency 병합 완료 → shape: (1143652, 7)
활동성 90일 병합 완료 → shape: (1143652, 8)
활동성 180일 병합 완료 → shape: (1143652, 9)
구매 간격 병합 완료 → shape: (1143652, 11)
다양성 병합 완료 → shape: (1143652, 13)
취향 병합 완료 → shape: (1143652, 15)
[v2] 구매 트렌드 병합 완료 → shape: (1143652, 17)
[v2] 다양성 변화 병합 완료 → shape: (1143652, 19)
[v2] 구매 리듬 병합 완료 → shape: (1143652, 21)
[NEW] 시퀀스 간격 t1~t5 병합 완료 → shape: (1143652, 26)
고객 메타 병합 완료 → shape: (1143652, 31)

최종 데이터 크기: (1143652, 31)


## 10-1. 누수 검증

In [15]:
n_unique = df_analysis['customer_id'].nunique()
n_total  = len(df_analysis)
assert n_unique == n_total, '중복 고객 존재!'
print(f'중복 없음 (총 {n_total:,}명)')

seq_features = [f'interval_t{i+1}' for i in range(N)]
print('\n[NEW] 시퀀스 피처 churn 상관관계 (절대값 0.95 이상이면 누수 의심)')
for col in seq_features:
    corr = df_analysis[col].corr(df_analysis['churn'])
    flag = '⚠️ 누수 의심' if abs(corr) > 0.95 else '✅ 정상'
    print(f'  {col:<15}: {corr:.4f}  {flag}')


중복 없음 (총 1,143,652명)

[NEW] 시퀀스 피처 churn 상관관계 (절대값 0.95 이상이면 누수 의심)
  interval_t1    : -0.0281  ✅ 정상
  interval_t2    : -0.0234  ✅ 정상
  interval_t3    : -0.0242  ✅ 정상
  interval_t4    : -0.0211  ✅ 정상
  interval_t5    : -0.0174  ✅ 정상


## 11. 데이터 저장

In [16]:
file_name = '../data/hnm_churn_dataset_v3.csv'
df_analysis.to_csv(file_name, index=False)

print(f'저장 완료: {file_name}')
print(f'최종 데이터 크기: {df_analysis.shape}')
print(f'컬럼 수: v2(26개) → v3({df_analysis.shape[1]}개) (+{df_analysis.shape[1]-26}개: interval_t1~t5)')


저장 완료: ../data/hnm_churn_dataset_v3.csv
최종 데이터 크기: (1143652, 31)
컬럼 수: v2(26개) → v3(31개) (+5개: interval_t1~t5)
